In [1]:
import pandas as pd

matchup_code = 'R6CH' # championship matchup
year = 2026

bracket_sims = pd.read_csv('./bracket_simulations.csv')
bracket_sims_w = pd.read_csv('./bracket_simulations_w.csv')

m_bracket_sims = bracket_sims[bracket_sims['Tournament'] == 'M']
w_bracket_sims = bracket_sims_w[bracket_sims_w['Tournament'] == 'W']

seeds_m = pd.read_csv("data/MNCAATourneySeeds.csv")
seeds_w = pd.read_csv("data/WNCAATourneySeeds.csv")

# Manually selecting outcome of first four games
seeds_m = pd.concat([seeds_m, pd.DataFrame([{'Season': 2026, 'Seed': 'X16', 'TeamID': 1250}])], ignore_index=True)
seeds_m = pd.concat([seeds_m, pd.DataFrame([{'Season': 2026, 'Seed': 'Y11', 'TeamID': 1275}])], ignore_index=True)
seeds_m = pd.concat([seeds_m, pd.DataFrame([{'Season': 2026, 'Seed': 'Y16', 'TeamID': 1224}])], ignore_index=True)
seeds_m = pd.concat([seeds_m, pd.DataFrame([{'Season': 2026, 'Seed': 'Z11', 'TeamID': 1301}])], ignore_index=True)

seeds_w = pd.concat([seeds_w, pd.DataFrame([{'Season': 2026, 'Seed': 'X10', 'TeamID': 3113}])], ignore_index=True)
seeds_w = pd.concat([seeds_w, pd.DataFrame([{'Season': 2026, 'Seed': 'X16', 'TeamID': 3359}])], ignore_index=True)
seeds_w = pd.concat([seeds_w, pd.DataFrame([{'Season': 2026, 'Seed': 'Y16', 'TeamID': 3283}])], ignore_index=True)
seeds_w = pd.concat([seeds_w, pd.DataFrame([{'Season': 2026, 'Seed': 'Z11', 'TeamID': 3304}])], ignore_index=True)

teams = pd.read_csv("data/MTeams.csv")
seeds_m = seeds_m.merge(teams[['TeamID', 'TeamName']], on='TeamID')
seeds_m = seeds_m[seeds_m['Season'] == year]

teams_w = pd.read_csv("data/WTeams.csv")
seeds_w = seeds_w.merge(teams_w[['TeamID', 'TeamName']], on='TeamID')
seeds_w = seeds_w[seeds_w['Season'] == year]

# Determining how many brackets were generated
num_brackets = bracket_sims['Bracket'].max()

# Labeling teams with the percentages of winning a particular matchup, given how many times they won in the simulation
value_counts = m_bracket_sims[m_bracket_sims['Slot'] == matchup_code]['Team'].value_counts()
percentages = (value_counts / num_brackets) * 100
team_names = seeds_m.set_index('Seed')['TeamName']
percentages.index = percentages.index.map(team_names)

# Represents the conditional probability that a team wins a particular matchup
percentages


Team
Duke              35.22
Michigan          21.29
Arizona           17.27
Florida            7.14
Houston            5.44
Iowa St            4.23
Purdue             1.96
Illinois           1.51
Connecticut        1.06
Arkansas           0.96
Gonzaga            0.91
Michigan St        0.80
Virginia           0.79
Kansas             0.26
St John's          0.20
Alabama            0.15
Nebraska           0.14
Clemson            0.09
Vanderbilt         0.09
Louisville         0.08
Tennessee          0.06
Texas Tech         0.06
UCLA               0.05
North Carolina     0.04
BYU                0.04
Wisconsin          0.03
Kentucky           0.02
Santa Clara        0.02
Akron              0.02
Miami FL           0.01
Iowa               0.01
St Louis           0.01
Ohio St            0.01
Utah St            0.01
McNeese St         0.01
St Mary's CA       0.01
Name: count, dtype: float64

In [2]:
# Labeling teams with the percentages of winning a particular matchup, given how many times they won in the simulation
value_counts = w_bracket_sims[w_bracket_sims['Slot'] == matchup_code]['Team'].value_counts()
percentages = (value_counts / num_brackets) * 100
team_names = seeds_w.set_index('Seed')['TeamName']
percentages.index = percentages.index.map(team_names)

# Represents the conditional probability that a team wins a particular matchup
percentages

Team
South Carolina    25.62
UCLA              23.97
Connecticut       22.99
Texas             19.46
LSU                6.44
Michigan           0.66
TCU                0.19
Iowa               0.13
Oklahoma           0.13
Louisville         0.12
Vanderbilt         0.12
Ohio St            0.09
Duke               0.05
North Carolina     0.02
Maryland           0.01
Name: count, dtype: float64

In [ ]:
# Identify projected upsets from predictions.csv (FIRST ROUND ONLY)
import re

# Load predictions
predictions = pd.read_csv('predictions.csv')

# Parse the ID to extract season and team IDs
predictions[['Season', 'T1_TeamID', 'T2_TeamID']] = predictions['ID'].str.split('_', expand=True)
predictions['Season'] = predictions['Season'].astype(int)
predictions['T1_TeamID'] = predictions['T1_TeamID'].astype(int)
predictions['T2_TeamID'] = predictions['T2_TeamID'].astype(int)

# Load round slots and filter for first round only
round_slots = pd.read_csv('data/MNCAATourneySlots.csv')
round_slots_m = round_slots[(round_slots['Season'] == 2026) & (round_slots['Slot'].str.contains('R1'))]

round_slots_w = pd.read_csv('data/WNCAATourneySlots.csv')
round_slots_w = round_slots_w[(round_slots_w['Season'] == 2026) & (round_slots_w['Slot'].str.contains('R1'))]

# Create seed to team mapping
m_seed_to_team = seeds_m[['Seed', 'TeamID']].drop_duplicates().set_index('Seed')['TeamID'].to_dict()
w_seed_to_team = seeds_w[['Seed', 'TeamID']].drop_duplicates().set_index('Seed')['TeamID'].to_dict()

# Get first round matchups as tuples of (team1, team2) for quick lookup
def get_first_round_matchups(round_slots_df, seed_to_team_dict):
    matchups = set()
    for _, row in round_slots_df.iterrows():
        strong_team = seed_to_team_dict.get(row['StrongSeed'])
        weak_team = seed_to_team_dict.get(row['WeakSeed'])
        if strong_team and weak_team:
            # Store as a frozenset so both orderings can be found
            matchups.add(frozenset([strong_team, weak_team]))
    return matchups

m_r1_matchups = get_first_round_matchups(round_slots_m, m_seed_to_team)
w_r1_matchups = get_first_round_matchups(round_slots_w, w_seed_to_team)

# Prepare seed data (add seeds for both team positions)
seeds_m_copy = seeds_m[['Season', 'TeamID', 'Seed']].copy()
seeds_m_copy.columns = ['Season', 'T1_TeamID', 'T1_Seed']
seeds_m_merged = predictions.merge(seeds_m_copy, on=['Season', 'T1_TeamID'], how='left')

seeds_m_copy2 = seeds_m[['Season', 'TeamID', 'Seed']].copy()
seeds_m_copy2.columns = ['Season', 'T2_TeamID', 'T2_Seed']
m_with_seeds = seeds_m_merged.merge(seeds_m_copy2, on=['Season', 'T2_TeamID'], how='left')

# Do the same for women
seeds_w_copy = seeds_w[['Season', 'TeamID', 'Seed']].copy()
seeds_w_copy.columns = ['Season', 'T1_TeamID', 'T1_Seed']
seeds_w_merged = predictions.merge(seeds_w_copy, on=['Season', 'T1_TeamID'], how='left')

seeds_w_copy2 = seeds_w[['Season', 'TeamID', 'Seed']].copy()
seeds_w_copy2.columns = ['Season', 'T2_TeamID', 'T2_Seed']
w_with_seeds = seeds_w_merged.merge(seeds_w_copy2, on=['Season', 'T2_TeamID'], how='left')

# Function to extract seed number from seed string (e.g., "W01" -> 1, "Y16b" -> 16)
def extract_seed_number(seed_str):
    if pd.isna(seed_str):
        return None
    match = re.search(r'(\d+)', str(seed_str))
    return int(match.group(1)) if match else None

# Extract seed numbers for men's tournament
m_with_seeds['T1_SeedNum'] = m_with_seeds['T1_Seed'].apply(extract_seed_number)
m_with_seeds['T2_SeedNum'] = m_with_seeds['T2_Seed'].apply(extract_seed_number)

# Extract seed numbers for women's tournament
w_with_seeds['T1_SeedNum'] = w_with_seeds['T1_Seed'].apply(extract_seed_number)
w_with_seeds['T2_SeedNum'] = w_with_seeds['T2_Seed'].apply(extract_seed_number)

# Identify upsets: both teams seeded, and the higher seed (underdog) has >= 50% probability
def find_upsets(df, tournament_name, r1_matchups):
    upsets = []
    for idx, row in df.iterrows():
        s1 = row['T1_SeedNum']
        s2 = row['T2_SeedNum']
        pred = row['Pred']
        t1_id = row['T1_TeamID']
        t2_id = row['T2_TeamID']
        
        # Skip if either team is not seeded
        if pd.isna(s1) or pd.isna(s2):
            continue
        
        # Skip if this matchup is not in the first round
        if frozenset([t1_id, t2_id]) not in r1_matchups:
            continue
        
        # Determine which team is the underdog (higher seed number = lower ranking)
        if s1 > s2:  # Team 1 is the underdog
            if pred >= 0.50:
                upsets.append({
                    'Tournament': tournament_name,
                    'UnderDog_TeamID': row['T1_TeamID'],
                    'UnderDog_Seed': row['T1_Seed'],
                    'UnderDog_SeedNum': s1,
                    'Favorite_TeamID': row['T2_TeamID'],
                    'Favorite_Seed': row['T2_Seed'],
                    'Favorite_SeedNum': s2,
                    'Upset_Probability': pred
                })
        elif s2 > s1:  # Team 2 is the underdog
            if (1 - pred) >= 0.50:
                upsets.append({
                    'Tournament': tournament_name,
                    'UnderDog_TeamID': row['T2_TeamID'],
                    'UnderDog_Seed': row['T2_Seed'],
                    'UnderDog_SeedNum': s2,
                    'Favorite_TeamID': row['T1_TeamID'],
                    'Favorite_Seed': row['T1_Seed'],
                    'Favorite_SeedNum': s1,
                    'Upset_Probability': 1 - pred
                })
    
    return pd.DataFrame(upsets)

# Find upsets for both tournaments (first round only)
m_upsets = find_upsets(m_with_seeds, 'Men', m_r1_matchups)
w_upsets = find_upsets(w_with_seeds, 'Women', w_r1_matchups)

# Combine and display
all_upsets = pd.concat([m_upsets, w_upsets], ignore_index=True)

# Merge with team names for better readability
teams_m_names = seeds_m[['TeamID', 'TeamName']].drop_duplicates()
teams_w_names = seeds_w[['TeamID', 'TeamName']].drop_duplicates()

# Add underdog team names
all_upsets = all_upsets.merge(teams_m_names.rename(columns={'TeamID': 'UnderDog_TeamID', 'TeamName': 'UnderDog_Name'}), 
                               on='UnderDog_TeamID', how='left')
all_upsets = all_upsets.merge(teams_w_names.rename(columns={'TeamID': 'UnderDog_TeamID', 'TeamName': 'UnderDog_Name'}), 
                               on='UnderDog_TeamID', how='left', suffixes=('', '_w'))
all_upsets['UnderDog_Name'] = all_upsets['UnderDog_Name'].fillna(all_upsets['UnderDog_Name_w'])
all_upsets = all_upsets.drop(columns=['UnderDog_Name_w'], errors='ignore')

# Add favorite team names
all_upsets = all_upsets.merge(teams_m_names.rename(columns={'TeamID': 'Favorite_TeamID', 'TeamName': 'Favorite_Name'}), 
                               on='Favorite_TeamID', how='left')
all_upsets = all_upsets.merge(teams_w_names.rename(columns={'TeamID': 'Favorite_TeamID', 'TeamName': 'Favorite_Name'}), 
                               on='Favorite_TeamID', how='left', suffixes=('', '_w'))
all_upsets['Favorite_Name'] = all_upsets['Favorite_Name'].fillna(all_upsets['Favorite_Name_w'])
all_upsets = all_upsets.drop(columns=['Favorite_Name_w'], errors='ignore')

# Sort by upset probability (highest first)
all_upsets = all_upsets.sort_values('Upset_Probability', ascending=False)

print(f"Total First Round Projected Upsets: {len(all_upsets)}\n")
all_upsets[['Tournament', 'UnderDog_Name', 'UnderDog_Seed', 'Favorite_Name', 'Favorite_Seed', 'Upset_Probability']]

Total Projected Upsets: 225



,Tournament,UnderDog_Name,UnderDog_Seed,Favorite_Name,Favorite_Seed,Upset_Probability
193,Women,Nebraska,Z11,Villanova,Z10,0.771626
192,Women,Nebraska,Z11a,Villanova,Z10,0.771626
145,Women,F Dickinson,X15,Howard,W14,0.745112
1,Men,Akron,Y12,Miami OH,Y11,0.738345
0,Men,Akron,Y12,Miami OH,Y11a,0.738345
...,...,...,...,...,...,...
48,Men,Vanderbilt,X05,Kansas,W04,0.504101
45,Men,Iowa,X09,Ohio St,W08,0.503886
100,Men,South Florida,W11,Texas A&M,X10,0.503750
21,Men,Iowa,X09,Georgia,Y08,0.502479


In [4]:
# Make a copy of bracket_simulations.csv FIRST, name it remaining_perfect_brackets.csv

def count_perfect_brackets(team_id, slot):
    """
    Count the number of perfect brackets remaining for a given team ID and slot.
    
    Parameters:
    team_id (int): The ID of the team.
    slot (str): The slot corresponding to a particular matchup.
    
    Returns:
    int: The number of perfect brackets remaining.
    """
    # Load the remaining perfect brackets
    remaining_brackets = pd.read_csv('remaining_perfect_brackets.csv')
    
    # Filter the bracket simulations for the given slot and team ID
    matching_brackets = m_bracket_sims[(m_bracket_sims['Slot'] == slot) & (m_bracket_sims['Team'] == team_id)]
    
    # Get the list of brackets that have the given team ID and slot
    matching_bracket_ids = matching_brackets['Bracket'].unique()
    
    # Remove brackets that don't have this pick from the remaining perfect brackets
    remaining_brackets = remaining_brackets[remaining_brackets['Bracket'].isin(matching_bracket_ids)]
    
    # Save the updated remaining perfect brackets
    remaining_brackets.to_csv('remaining_perfect_brackets.csv', index=False)
    
    # Return the count of remaining perfect brackets
    return remaining_brackets['Bracket'].nunique()

team_id = ''  # Replace with the desired team seed identifier
slot = 'R1W8'  # Replace with the desired slot
perfect_brackets = 0

try:
    assert(len(team_id) > 0)
    assert(len(slot) > 0)
    perfect_brackets = count_perfect_brackets(team_id, slot)
    print(f"Percent of perfect brackets remaining for team ID {team_id} in slot {slot}: {(perfect_brackets / num_brackets)*100}% ({perfect_brackets} brackets)")
except:
    remaining_brackets = pd.read_csv('remaining_perfect_brackets.csv')
    print(f"Percent of perfect brackets remaining: {(remaining_brackets['Bracket'].nunique() / num_brackets)*100}% ({remaining_brackets['Bracket'].nunique()} brackets)")    
    print()
    perfect_brackets = 0

FileNotFoundError: [Errno 2] No such file or directory: 'remaining_perfect_brackets.csv'